<a href="https://colab.research.google.com/github/Leo278V/Final-Assignment-PDS/blob/Final-Assignment-V2/NLP_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [178]:
import pandas as pd

In [179]:
data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/linkedin_experience_annotated.csv"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [180]:
#Fill active Jobs with current date
from datetime import date


df_profiles["endDate"] = df_profiles["endDate"].astype(str)

df_profiles.loc[
    (df_profiles["status"] == "ACTIVE") & (df_profiles["endDate"].isin(["nan", "NaT"])),
    "endDate"
] = date.today().strftime("%Y-%m")

In [181]:
#Remove linkedIN column

df_profiles.drop(columns=['linkedin'], inplace=True, errors='ignore')
df_profiles.head()

,organization,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0


In [182]:
df_profiles.isna().sum()


,0
organization,0
position,0
startDate,118
endDate,0
status,0
department,0
seniority,0
person_id,0


Feature Encoding

In [183]:
#Organization encoding - Frequency Encoding

org_freq = df_profiles["organization"].value_counts(normalize=True)
df_profiles["organization_freq"] = df_profiles["organization"].map(org_freq)

df_profiles = df_profiles.drop(columns=["organization"])


In [184]:
#Position Encoding - Sentence Embeddings

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
X_position = model.encode(df_profiles["position"].fillna("").tolist())


In [185]:
#startDate & endDate -> Feature Engineering → job_duration_years
start = pd.to_datetime(df_profiles["startDate"], format="%Y-%m", errors='coerce')
end   = pd.to_datetime(df_profiles["endDate"],   format="%Y-%m", errors='coerce')

df_profiles["job_duration_years"] = (end - start).dt.days / 365

df_profiles = df_profiles.drop(columns=["startDate", "endDate"])

In [186]:
#status → Binary Encoding
df_profiles["status_bin"] = df_profiles["status"].map({"ACTIVE": 1, "INACTIVE": 0})
df_profiles = df_profiles.drop(columns=["status"])


In [187]:
df_profiles.head()

,position,department,seniority,person_id,organization_freq,job_duration_years,status_bin
0,Prokurist,Other,Management,0,0.001895,6.339726,1.0
1,CFO,Other,Management,0,0.001895,6.424658,1.0
2,Betriebswirtin,Other,Professional,0,0.001895,6.424658,1.0
3,Prokuristin,Other,Management,0,0.001895,6.424658,1.0
4,CFO,Other,Management,0,0.001895,6.424658,1.0


In [188]:
df_seniority_raw = df_profiles[[
    "position",
    "department",
    "seniority",
    "person_id",
    "organization_freq",
    "job_duration_years",
    "status_bin",
]].copy()



**Datframes for Seniority and Department**

In [189]:
from sklearn.preprocessing import LabelEncoder

seniority_map = {
    "Junior": 0,
    "Professional": 1,
    "Senior": 2,
    "Lead": 3,
    "Management": 4
}

df_seniority_encoded = df_seniority_raw[[
    "position",
    "department",
    "seniority",
    "person_id",
    "organization_freq",
    "job_duration_years",
    "status_bin",
]].copy()

# Target (Ordinal)
df_seniority_encoded["seniority"] = df_seniority_encoded["seniority"].map(seniority_map)


# Department als Feature (Label Encoding)
le_dept = LabelEncoder()
df_seniority_encoded["department_enc"] = le_dept.fit_transform(
    df_seniority_encoded["department"]
)

df_seniority_encoded = df_seniority_encoded.drop(columns=["department"])

In [190]:
df_seniority_encoded.isna().sum()


,0
position,0
seniority,141
person_id,0
organization_freq,0
job_duration_years,348
status_bin,118
department_enc,0


In [191]:
#Department Dataframe for Predicting Department Model
df_department_raw = df_profiles[[
    "position",
    "department",
    "seniority",
    "person_id",
    "organization_freq",
    "job_duration_years",
    "status_bin",
]].copy()

df_department_encoded = df_profiles[[
    "position",
    "department",
    "seniority",
    "person_id",
    "organization_freq",
    "job_duration_years",
    "status_bin",
]].copy()

# Target (Label Encoding)
le_dep = LabelEncoder()
df_department_encoded["department"] = le_dep.fit_transform(
    df_department_encoded["department"]
)

# Seniority als Feature (Ordinal)
df_department_encoded["seniority_ord"] = df_department_encoded["seniority"].map(seniority_map)
df_department_encoded = df_department_encoded.drop(columns=["seniority"])


Model Predicting Seniority

In [192]:
y = df_seniority_encoded['seniority']

In [193]:
X = df_seniority_encoded.drop('seniority', axis=1)

In [194]:
from sklearn.model_selection import train_test_split


In [195]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)


In [196]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [197]:
text_features = "position"
numeric_features = [
    "organization_freq",
    "job_duration_years",
    "status_bin",
    "department_enc"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            max_features=500,
            ngram_range=(1, 2),
            stop_words="english"
        ), text_features),
        ("num", StandardScaler(), numeric_features)
    ]
)


In [198]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            multi_class="auto"
        ))
    ]
)
